# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

### Realizo la carga de todos los archivos .html. Y como se descargaron los archivos con la extensión .htm. Use la expresión: *.hmt *

In [2]:
from bs4 import BeautifulSoup
import glob

soups = []

for file in glob.glob("data/*.htm*"):
    with open(file, "r", encoding="utf-8") as f:
        html_content = f.read()
        soup = BeautifulSoup(html_content, "html.parser")
        soups.append(soup)

print(f"Se cargaron {len(soups)} archivos HTML.")

Se cargaron 15 archivos HTML.


In [3]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Zesty Slow Cooker Chicken Barbecue'

In [5]:
from bs4 import BeautifulSoup
import glob

# Recorre todos los HTML de la carpeta data
for archivo in glob.glob("data/*.htm*"):

    print("=" * 60)
    print(f"Archivo: {archivo}")

    with open(archivo, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    # Nombre de la receta (opcional)
    titulo = soup.find("h1")
    if titulo:
        print(f"\nReceta: {titulo.get_text(strip=True)}")

    # Ingredientes
    print("\nIngredientes:")

    ingredientes = soup.find_all(
        "li",
        class_="mm-recipes-structured-ingredients__list-item"
    )

    for ingrediente in ingredientes:
        print("-", ingrediente.get_text(" ", strip=True))

    print()

Archivo: data\Air-Fryer-BBQ-Baby-Back-Ribs-Recipe.htm

Receta: Air Fryer BBQ Baby Back Ribs

Ingredientes:
- 3 pounds baby back pork ribs
- 1 tablespoon brown sugar
- 1 tablespoon white sugar
- 1 teaspoon sweet paprika
- 1 teaspoon smoked paprika
- 1 teaspoon granulated garlic
- ½ teaspoon ground black pepper
- ½ teaspoon ground cumin
- ½ teaspoon granulated onion
- ¼ teaspoon Greek seasoning (Optional)
- ⅓ cup barbeque sauce

Archivo: data\Baked-BBQ-Chicken-Drumsticks-Recipe.htm

Receta: Baked BBQ Chicken Drumsticks

Ingredientes:
- 1 ½ pounds chicken drumsticks
- ¼ cup extra-virgin olive oil
- 1 teaspoon salt
- ½ teaspoon freshly ground black pepper
- ½ teaspoon paprika
- ¼ teaspoon garlic powder
- ¼ teaspoon onion powder
- ¼ teaspoon cayenne pepper
- 1 ½ cups barbeque sauce (such as Sweet Baby Ray's), or more to taste

Archivo: data\BBQ-Chicken-Breasts-in-the-Oven-Recipe.htm

Receta: BBQ Chicken Breasts in the Oven

Ingredientes:
- 4 whole skin-on, bone-in chicken breasts
- 1 cup ba

## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [7]:
import glob
import os
from bs4 import BeautifulSoup

# Buscar todos los archivos con extensión .html y .htm en el directorio actual
# Si están en una carpeta específica, puedes poner ej: 'recetas/*.htm*'
archivos_recetas = glob.glob("data/*.htm*")

print(f"Se encontraron {len(archivos_recetas)} archivos de recetas para procesar.\n")

for ruta_archivo in archivos_recetas:
    # Obtener solo el nombre del archivo para mostrar en el print
    nombre_archivo = os.path.basename(ruta_archivo)
    print("=" * 60)
    print(f"PROCESANDO ARCHIVO: {nombre_archivo}")
    print("=" * 60)
    
    # Leer el contenido del archivo HTML
    with open(ruta_archivo, 'r', encoding='utf-8') as file:
        content = file.read()
        
    soup = BeautifulSoup(content, 'html.parser')
    
    # 1. Extraer el Título (asumiendo que ya tienes la variable 'title' definida previamente)
    # Por si acaso, lo extraemos aquí de la etiqueta <h1> o de <title>
    title_tag = soup.find("h1")
    title = title_tag.get_text().strip() if title_tag else soup.title.get_text().strip() if soup.title else "Sin Título"

    # 2. Extraer la Descripción (con validación por si no existe la etiqueta meta)
    meta_desc = soup.find("meta", {"name": "description"})
    description = meta_desc["content"].strip() if meta_desc and meta_desc.has_attr("content") else "Sin descripción disponible"

    # 3. Extraer los Ingredientes
    ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

    # 4. Extraer las Instrucciones
    instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [instruction.get_text().strip() for instruction in instructions_section]

    # 5. Extraer la Información Nutricional
    nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
    nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

    # --- Imprimir la información extraída del archivo actual ---
    print(f"Recipe Title: {title}")
    print(f"Description: {description}")
    
    print("\nIngredients:")
    if ingredients:
        for ingredient in ingredients:
            print("-", ingredient)
    else:
        print("(No se encontraron ingredientes)")
        
    print("\nInstructions:")
    if instructions:
        for i, instruction in enumerate(instructions, 1):
            print(f"{i}. {instruction}")
    else:
        print("(No se encontraron instrucciones)")
        
    print("\nNutrition Facts:")
    if nutrition_facts:
        for fact in nutrition_facts:
            print("-", fact)
    else:
        print("(No se encontró información nutricional)")
        
    print("\n" + "#" * 60 + "\n") # Separador entre recetas

Se encontraron 15 archivos de recetas para procesar.

PROCESANDO ARCHIVO: Air-Fryer-BBQ-Baby-Back-Ribs-Recipe.htm
Recipe Title: Air Fryer BBQ Baby Back Ribs
Description: These air fryer ribs are juicy, finger-licking delicious with mouthwatering flavor thanks to an easy homemade spice rub and barbeque glaze.

Ingredients:
- 3 pounds baby back pork ribs
- 1 tablespoon brown sugar
- 1 tablespoon white sugar
- 1 teaspoon sweet paprika
- 1 teaspoon smoked paprika
- 1 teaspoon granulated garlic
- ½ teaspoon ground black pepper
- ½ teaspoon ground cumin
- ½ teaspoon granulated onion
- ¼ teaspoon Greek seasoning (Optional)
- ⅓ cup barbeque sauce

Instructions:
1. Preheat an air fryer to 350 degrees F (175 degrees C).
2. Strip membrane from the back of ribs. Cut ribs into 4 equal portions.
3. Combine brown sugar, white sugar, sweet paprika, smoked paprika, granulated garlic, pepper, cumin, onion, and Greek seasoning in a small bowl.
4. Rub spice mixture all over ribs. Place ribs in the air fry

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [ ]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# social_domains = {
#     "facebook.com", "instagram.com", "pinterest.com", "tiktok.com",
#     "youtube.com", "twitter.com", "x.com", "linkedin.com", "flipboard.com"
# }

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [ ]:
!pip install google-genai

In [8]:
import os
import glob
from bs4 import BeautifulSoup
from google import genai
from google.genai import types

# Construir el Corpus de Datos desde los archivos HTML

corpus_recetas = []

# Iterar sobre todos los archivos en la carpeta data/ como en las celdas previas
for file_path in glob.glob("data/*.htm*"):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
        
    # Extraer campos clave de forma segura
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else os.path.basename(file_path)
    
    meta_desc = soup.find("meta", {"name": "description"})
    description = meta_desc["content"].strip() if meta_desc and meta_desc.has_attr("content") else "Sin descripción"
    
    ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [ing.get_text(" ", strip=True) for ing in ingredients_section]
    
    instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [ins.get_text().strip() for ins in instructions_section]
    
    nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
    nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]
    
    # Formatear la receta como un bloque de texto estructurado
    receta_texto = f"RECETA: {title}\n"
    receta_texto += f"Descripción: {description}\n"
    receta_texto += "Ingredientes:\n" + "\n".join([f"- {ing}" for ing in ingredients]) + "\n"
    receta_texto += "Instrucciones:\n" + "\n".join([f"{i+1}. {ins}" for i, ins in enumerate(instructions)]) + "\n"
    receta_texto += "Información Nutricional:\n" + "\n".join([f"- {fct}" for fct in nutrition_facts]) + "\n"
    
    corpus_recetas.append(receta_texto)

# Unir todo el conocimiento recolectado en una sola variable de contexto
contexto_rag = "\n" + "="*40 + "\n".join(corpus_recetas)
print(f"Corpus indexado con éxito. Total de recetas listas para RAG: {len(corpus_recetas)}")

Corpus indexado con éxito. Total de recetas listas para RAG: 15


In [ ]:
# Uso de Gemini API
import os
from dotenv import load_dotenv

load_dotenv() # Carga las variables desde el archivo .env

client = genai.Client()

prompt_sistema = """
Eres un asistente experto en cocina y nutrición que responde basándose exclusivamente en el corpus de recetas provisto.
Tu objetivo es ayudar al usuario a encontrar recetas, sugerir platos basados en ingredientes disponibles o calcular datos nutricionales.

Reglas críticas:
1. Si la respuesta no se puede deducir a partir del corpus de recetas adjunto, responde amablemente indicando que no posees esa información.
2. No inventes ingredientes ni pasos de preparación que no aparezcan explícitamente en el texto.
3. Sé preciso y estructurado en tus respuestas.
"""

# Consulta sobre recetas
consulta_usuario = "¿Qué recetas puedo preparar si solo tengo pechugas de pollo (chicken breast) y azúcar rubia (brown sugar)?"

# Construir el contenido combinando el Corpus y la Pregunta
contents = f"""
[CONTEXTO DE RECETAS DISPONIBLES]
{contexto_rag}
[FIN DEL CONTEXTO]

Pregunta del usuario: {consulta_usuario}
"""

In [13]:
# Paso C: Ejecución de la consulta RAG

print(f"\nRealizando consulta RAG: '{consulta_usuario}'...\n")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=prompt_sistema,
        temperature=0.2 # Temperatura baja para mayor fidelidad al texto y menos alucinaciones
    )
)

print("Respuesta de Gemini (RAG):")
print("-" * 40)
print(response.text)


Realizando consulta RAG: '¿Qué recetas puedo preparar si solo tengo pechugas de pollo (chicken breast) y azúcar rubia (brown sugar)?'...

Respuesta de Gemini (RAG):
----------------------------------------
Basándome exclusivamente en el corpus de recetas proporcionado, puedes preparar las siguientes recetas si solo tienes pechugas de pollo y azúcar rubia:

1.  **Slow Cooker Barbecue Chicken Breast**
2.  **Zesty Slow Cooker Chicken Barbecue**
